In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install langchain openai faiss-cpu tiktoken pypdf chromadb
!pip install beautifulsoup4 requests
!pip install streamlit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 53.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 58.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6

In [ ]:
import os
from pathlib import Path

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

import os
import uuid
import chromadb
import numpy as np
from typing import List, Any

from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage, SystemMessage





In [ ]:

def process_all_pdfs(pdf_directory):
    all_documents = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")

        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            for doc in documents:
                doc.metadata["source_file"] = pdf_file.name
                doc.metadata["file_type"] = "pdf"

            all_documents.extend(documents)
            print(f"  Loaded {len(documents)} pages")

        except Exception as e:
            print(f"  Error: {e}")

    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents
all_pdf_documents = process_all_pdfs("/content/drive/MyDrive/Purplemerit/Data/")

Found 1 PDF files to process

Processing: schedule.pdf
  Loaded 2 pages

Total documents loaded: 2


In [ ]:
all_pdf_documents

[Document(metadata={'producer': 'Mac OS X 10.9.1 Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20140206034040Z00'00'", 'title': 'Microsoft Word - Proposal for UG Curriculum of CSE Dept v2.2.doc', 'moddate': "D:20140206034040Z00'00'", 'source': '/content/drive/MyDrive/Purplemerit/Data/schedule.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'schedule.pdf', 'file_type': 'pdf'}, page_content='! 4!\nB.Tech!CSE!Semester-wise Course Schedule: (Credits completed in 1st year: 34 Credits of Institute Core)!Sem Lecture Course I Lecture Course II Lecture Course III Lecture Course IV Lecture CourseV other L-T-P/Cr III Discrete Structures 3-1-0 [DC] \nDigital Logic & Design 3-0-4 [DC] \nData Structures 3-0-4 [DC] \nMaterial Science [Physics]  3 [PLES2] \nProbability and Stochastic Processes 3-1-0 [PLES1] \nCSN100 0-0-2 Non-Graded  21  \nIV Prog. Languages 3-0-4 [DC] \nComputer Architecture 3-0-2 [DC] \nEEL205: Signals and Systems  3-1-0 [PLES3] \nEnv  (2-0-0) HU1 4 

In [ ]:
####Chunking############


from langchain_text_splitters import RecursiveCharacterTextSplitter


def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )

    split_docs = text_splitter.split_documents(documents)

    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")


    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs



chunked_documents = split_documents(documents)

Split 4 documents into 45 chunks

Example chunk:
Content: All Courses – Computer Science and Engineering
COL100 Introdution to Computer Science
4 credits (3-0-2) Organization of Computing Systems. Concept of an algorithm; termination and correctness. Algorit...
Metadata: {'source': 'courses.html'}


In [ ]:
chunks=split_documents(documents)
chunks

Split 4 documents into 45 chunks

Example chunk:
Content: All Courses – Computer Science and Engineering
COL100 Introdution to Computer Science
4 credits (3-0-2) Organization of Computing Systems. Concept of an algorithm; termination and correctness. Algorit...
Metadata: {'source': 'courses.html'}


[Document(metadata={'source': 'courses.html'}, page_content='All Courses – Computer Science and Engineering\nCOL100 Introdution to Computer Science\n4 credits (3-0-2) Organization of Computing Systems. Concept of an algorithm; termination and correctness. Algorithms to programs: specification, top-down development and stepwise refinement. Problem solving using a functional style; Correctness […]read more\nCOL106 Data Structures & Algorithms\n5 credits (3-0-4) Pre-requisites: COL100 Introduction to object-oriented programming through stacks queues and linked lists. Dictionaries; skip-lists, hashing, analysis of collision resolution techniques. Trees, traversals, binary search trees, optimal and […]read more\nCOL202 Discrete Mathematical Structures\n4 credits (3-1-0) Overlaps with: MTL180 Propositional logic, Predicate Calculus and Quantifiers; Proof Methods; Sets, functions, relations, Cardinality, Infinity and Diagonalization; Induction and Recursion; Modular Arithmetic, Euclid’s Algor

In [ ]:
###embedding And vectorStoreDB###

In [ ]:

class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initialize the embedding manager
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generated embeddings for a list of texts

        """
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings



embedding_manager = EmbeddingManager()

# converting chunks to text
texts = [doc.page_content for doc in chunked_documents]



# Generating embeddings
embeddings = embedding_manager.generate_embeddings(texts)

Loading embedding model: all-MiniLM-L6-v2


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully. Embedding dimension: 384
Generating embeddings for 45 texts...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Generated embeddings with shape: (45, 384)


In [ ]:
##Vector embedding

# -------------------------------
# VECTOR STORE CLASS
# -------------------------------
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """
        Initialize the vector store
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise



vectorstore = VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 0


In [ ]:
chunks

[Document(metadata={'source': 'courses.html'}, page_content='All Courses – Computer Science and Engineering\nCOL100 Introdution to Computer Science\n4 credits (3-0-2) Organization of Computing Systems. Concept of an algorithm; termination and correctness. Algorithms to programs: specification, top-down development and stepwise refinement. Problem solving using a functional style; Correctness […]read more\nCOL106 Data Structures & Algorithms\n5 credits (3-0-4) Pre-requisites: COL100 Introduction to object-oriented programming through stacks queues and linked lists. Dictionaries; skip-lists, hashing, analysis of collision resolution techniques. Trees, traversals, binary search trees, optimal and […]read more\nCOL202 Discrete Mathematical Structures\n4 credits (3-1-0) Overlaps with: MTL180 Propositional logic, Predicate Calculus and Quantifiers; Proof Methods; Sets, functions, relations, Cardinality, Infinity and Diagonalization; Induction and Recursion; Modular Arithmetic, Euclid’s Algor

In [ ]:
### Convert the text to embeddings
texts = [doc.page_content for doc in chunks]

## Generate the Embeddings
embeddings = embedding_manager.generate_embeddings(texts)

## Store in the vector database
vectorstore.add_documents(chunks, embeddings)

Generating embeddings for 45 texts...


Batches:   0%|          | 0/2 [00:00<?, ?it/s]

Generated embeddings with shape: (45, 384)
Adding 45 documents to vector store...
Successfully added 45 documents to vector store
Total documents in collection: 45


In [ ]:
##Retrieve pipeline from vector store
class RAGRetriever:


    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )


            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")

            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever = RAGRetriever(vectorstore, embedding_manager)

In [ ]:
rag_retriever

In [ ]:
rag_retriever.retrieve("What are prerequisites for COL106?")

Retrieving documents for query: 'What are prerequisites for COL106?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)


[{'id': 'doc_c9f06f8b_13',
  'content': 'COL740 Software Engineering\n4 credits (3-0-2) Pre-requisites: COL106, COL226 Introduction to Software Engineering, Software Life Cycle models and Processes, Requirement Engineering, System Models, Architectural Design, Abstraction & Modularity, Structured Programming, Object-oriented techniques, Design […]read more\nCOD745 Minor Project\n3 credits (0-0-6) Pre-requisites: 60 Research and development oriented projects based on problems of practical and theoretical interest. Evaluation done based on periodic presentations, student seminars, written reports, and evaluation […]read more\nCOP745 Digital System Design Laboratory\n3 credits (0-0-6) Pre-requisites: COL215 OR Equivalent Being primarily a laboratory course, it would consist of a series of assignments that would increase in complexity in terms of designs to […]read more\nCOL750 Foundations of Automatic Verification',
  'metadata': {'doc_index': 13,
   'content_length': 877,
   'source': 'c

In [ ]:
!pip install langchain-groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.5 MB/s eta 0:00:00


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"] = "your_key"
print(os.getenv("GROQ_API_KEY"))

your_key


In [ ]:


class GroqLLM:
    def __init__(self, model_name: str = "llama-3.1-8b-instant", api_key: str =None):
        """
        Initialize Groq LLM
        """
        self.model_name = model_name
        self.api_key = api_key or os.environ.get("GROQ_API_KEY")

        if not self.api_key:
            raise ValueError("Groq API key is required. Set GROQ_API_KEY environment variable or pass api_key parameter.")

        self.llm = ChatGroq(
            groq_api_key=self.api_key,
            model_name=self.model_name,
            temperature=0.1,
            max_tokens=1024
        )

        print(f"Initialized Groq LLM with model: {self.model_name}")

    def generate_response(self, query: str, context: str, max_length: int = 500) -> str:
        """
        Generate response using retrieved context
        """

        #prompt template
        prompt_template = PromptTemplate(
            input_variables=["context", "question"],
            template="""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

Context:
{context}

Question: {question}

Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""
        )

        # Format the prompt
        formatted_prompt = prompt_template.format(context=context, question=query)

        try:
            # Generate response
            messages = [HumanMessage(content=formatted_prompt)]
            response = self.llm.invoke(messages)
            return response.content

        except Exception as e:
            return f"Error generating response: {str(e)}"

    def generate_response_simple(self, query: str, context: str) -> str:
        """
        Simple response generation without complex prompting
        """
        simple_prompt = f"""Based on this context: {context}

Question: {query}

Answer:"""

        try:
            messages = [HumanMessage(content=simple_prompt)]
            response = self.llm.invoke(messages)
            return response.content
        except Exception as e:
            return f"Error: {str(e)}"

In [ ]:
# Initialize Groq LLM
import os

try:
    groq_llm = GroqLLM(api_key=os.getenv("GROQ_API_KEY"))
    print("Groq LLM initialized successfully!")
except ValueError as e:
    print(f"Warning: {e}")
    print("Please set your GROQ_API_KEY environment variable to use the LLM.")
    groq_llm = None

Initialized Groq LLM with model: llama-3.1-8b-instant
Groq LLM initialized successfully!


In [ ]:
rag_retriever.retrieve("Unified Multi-task Learning Framework")

Retrieving documents for query: 'Unified Multi-task Learning Framework'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 documents (after filtering)


[]

In [ ]:
### Simple RAG pipeline with Groq LLM
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()

### Initialize the Groq LLM
groq_api_key = os.getenv("GROQ_API_KEY")

llm = ChatGroq(
    groq_api_key=groq_api_key,
    model_name="llama-3.1-8b-instant",
    temperature=0.1,
    max_tokens=1024
)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):
    ## retriever the context
    results = retriever.retrieve(query, top_k=top_k)
    context = "\n\n".join([doc['content'] for doc in results]) if results else ""

    if not context:
        return "No relevant context found to answer the question."

    ## generate the answwer using GROQ LLM
    prompt = f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""

    response = llm.invoke([prompt.format(context=context, query=query)])
    return response.content

In [ ]:
answer = rag_simple("What are prerequisites for COL226?", rag_retriever, llm ,top_k=5)
print(answer)

Retrieving documents for query: 'What are prerequisites for COL226?'
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents (after filtering)
COL106.


In [ ]:
from typing import List, Dict, Any

def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)

    if not results:
        return {
            'answer': 'No relevant context found.',
            'sources': [],
            'confidence': 0.0,
            'context': ''
        }

    # Preparing context and sources
    context = "\n\n".join([doc['content'] for doc in results])

    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]

    confidence = max([doc['similarity_score'] for doc in results])

    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.

Context:
{context}

Question: {query}

Answer:"""

    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }

    if return_context:
        output['context'] = context

    return output


result = rag_advanced(
    "What topics are covered in COL202?",
    rag_retriever,
    llm,
    top_k=3,
    min_score=0.1,
    return_context=True
)

print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

Retrieving documents for query: 'What topics are covered in COL202?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
Answer: There is no information about COL202 in the given context. The provided context only mentions courses COL868, COL869, and COL870.
Sources: [{'source': 'courses.html', 'page': 'unknown', 'score': 0.1394639015197754, 'preview': 'COL868 Special topics in Database Systems\n3 credits (3-0-0) Pre-requisites: COL334 / COL672 / Equivalent The contents would include specific advanced topics in Database Management Systems in which research is currently going on in the department. […]read more\nCOL869 Special topics in Concurrency\n3 c...'}]
Confidence: 0.1394639015197754
Context Preview: COL868 Special topics in Database Systems
3 credits (3-0-0) Pre-requisites: COL334 / COL672 / Equivalent The contents would include specific advanced topics in Database Management Systems in which research is currently going on in the department. […]read more
COL869 Special topics in Concurrency
3 c


In [ ]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time


class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)

        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])

            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]

            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.
Context:
{context}

Question: {question}

Answer:"""

            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()

            response = self.llm.invoke([prompt.format(context=context, question=question)])
            answer = response.content


        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer


        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content


        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }



adv_rag = AdvancedRAGPipeline(rag_retriever, llm)

result = adv_rag.query(
    "What topics are covered in COL202?",
    top_k=3,
    min_score=0.1,
    stream=True,
    summarize=True
)

print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])

Retrieving documents for query: 'What topics are covered in COL202?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Generated embeddings with shape: (1, 384)
Retrieved 1 documents (after filtering)
Streaming answer:
Use the following context to answer the question concisely.
Context:
COL868 Special topics in Database Systems
3 credits (3-0-0) Pre-requisites: COL334 / COL672 / Equivalent The contents would include specific advanced topics in Database Management Systems in which research is currently going on in the department. […]read more
COL869 Special topics in Concurrency
3 credits (3-0-0) The course will focus on research issues in concurrent, distributed and mobile computations. Models of Concurrent, Distributed and Mobile computation. Process calculi, Event Structures, Petri Nets an […]read more
COL870 Special Topics in Machine Learning
3 credits (3-0-0) Pre-requisites: COL341 OR Equivalent Contents may vary based on the instructor’s expertise and interests within the broader area of Machine Learning. Example topics include (but not limiting […]read more
COL871 Special Topics in programming la